### I. Importing Stuff

In [1]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.models import convnext_base, ConvNeXt_Base_Weights

from PIL import Image
import ast

from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


### II. Loading Metadata 

In [3]:
split_mode = "closed"
segment = "head"

DATASET_DIR = "./data/turtle-data"
METADATA_FILE = Path(DATASET_DIR) / f"metadata_splits_{segment}.csv"

df = pd.read_csv(METADATA_FILE)
df["file_name"] = df["file_name"].apply(lambda x: str(Path(DATASET_DIR) / x))
df = df[["file_name", "identity", f"split_{split_mode}", "bounding_box"]].copy()

identities = df["identity"].unique().tolist()
label_to_index = {id_: i for i, id_ in enumerate(identities)}
index_to_label = {i: id_ for id_, i in label_to_index.items()}

df["label"] = df["identity"].map(label_to_index).astype(int)

### III. Dataset

In [4]:
class SeaTurtleDataset(Dataset):
    def __init__(self, df, split_mode, split, transform=None, use_bbox_crop=True):
        self.df = df[df[f"split_{split_mode}"] == split].reset_index(drop=True)
        self.transform = transform
        self.use_bbox_crop = use_bbox_crop

    def __len__(self):
        return len(self.df)
    
    def _crop_bbox(self, img, bbox):
        if bbox is None or (isinstance(bbox, float) and np.isnan(bbox)):
            return img
        if isinstance(bbox, str):
            bbox = ast.literal_eval(bbox)
        if not (isinstance(bbox, (list, tuple)) and len(bbox) in (4,)):
            return img
        w, h = img.size
        x1, y1, x2, y2 = None, None, None, None
        if bbox[2] > 0 and bbox[3] > 0 and (bbox[0] + bbox[2] <= w + 5) and (bbox[1] + bbox[3] <= h + 5):
            x, y, bw, bh = bbox
            x1, y1, x2, y2 = x, y, x + bw, y + bh
        else:
            x1, y1, x2, y2 = bbox
        x1 = int(max(0, min(w - 1, x1)))
        y1 = int(max(0, min(h - 1, y1)))
        x2 = int(max(1, min(w, x2)))
        y2 = int(max(1, min(h, y2)))
        if x2 <= x1 or y2 <= y1:
            return img
        return img.crop((x1, y1, x2, y2))
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["file_name"]
        label = int(row["label"])
        bbox = row["bounding_box"]
        img = Image.open(img_path).convert("RGB")
        if self.use_bbox_crop:
            img = self._crop_bbox(img, bbox)
        if self.transform is not None:
            img = self.transform(img)
        return img, label

### IV. Transforms and Dataloaders

In [5]:
transforms_train = T.Compose([
    T.Resize((224, 224)),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

transforms_test = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

train_set = SeaTurtleDataset(df, split_mode, "train", transform=transforms_train, use_bbox_crop=True)
val_set   = SeaTurtleDataset(df, split_mode, "valid", transform=transforms_test, use_bbox_crop=True)
test_set  = SeaTurtleDataset(df, split_mode, "test",  transform=transforms_test, use_bbox_crop=True)

print("Train/Val/Test:", len(train_set), len(val_set), len(test_set))

BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = True
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

num_classes = len(identities)
print("num_classes:", num_classes)


Train/Val/Test: 4571 1380 2575
num_classes: 438


### V. ConvNeXt backbone and AdaFace head

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import convnext_base, convnext_small, ConvNeXt_Base_Weights, ConvNeXt_Small_Weights

class ConvNeXtBackbone(nn.Module):
    def __init__(self, embedding_dim=512, pretrained=True, dropout=0.1):
        super().__init__()
        weights = ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
        model = convnext_small(weights=weights)
        
        in_dim = model.classifier[2].in_features
        model.classifier[2] = nn.Identity()
        self.backbone = model
        
        # Projection with dropout
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(in_dim, embedding_dim)
        
        # BN neck (crucial for metric learning)
        self.bn = nn.BatchNorm1d(embedding_dim)
        self.bn.bias.requires_grad_(False)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        emb = self.proj(feat)
        emb = self.bn(emb)
        # Return raw embeddings (not normalized)
        return emb


class AdaFace(nn.Module):
    """
    Proper AdaFace implementation with quality-adaptive margins
    """
    def __init__(self, embedding_size, num_classes, s=64.0, m=0.4, h=0.333, t_alpha=0.01, eps=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.s = s
        self.m = m
        self.h = h
        self.t_alpha = t_alpha
        self.eps = eps

        self.weight = nn.Parameter(torch.randn(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)

        # Running statistics of feature norms
        self.register_buffer("batch_mean", torch.tensor(20.0))
        self.register_buffer("batch_std", torch.tensor(100.0))

    @torch.no_grad()
    def _update_norm_stats(self, norms):
        if self.training:
            mean = norms.mean()
            std = norms.std().clamp_min(self.eps)
            self.batch_mean = (1 - self.t_alpha) * self.batch_mean + self.t_alpha * mean
            self.batch_std  = (1 - self.t_alpha) * self.batch_std  + self.t_alpha * std

    def forward(self, embeddings, labels=None):
        # Compute norms BEFORE normalization (quality indicator)
        norms = torch.norm(embeddings, p=2, dim=1, keepdim=True).clamp(min=self.eps)
        
        # Normalize embeddings and weights
        embeddings_norm = embeddings / norms
        embeddings_norm = embeddings_norm.squeeze()
        norms = norms.squeeze()
        
        weight_norm = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings_norm, weight_norm)
        cosine = cosine.clamp(-1 + self.eps, 1 - self.eps)

        if labels is None:
            # Inference mode
            return cosine * self.s, embeddings_norm

        # Update norm statistics
        self._update_norm_stats(norms)
        
        # Adaptive margin based on feature quality (norm)
        margin_scaler = (norms - self.batch_mean) / (self.batch_std + self.eps)
        margin_scaler = margin_scaler * self.h
        margin_scaler = margin_scaler.clamp(-1, 1)
        
        # Build one-hot
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        
        # Angular margin (g_angular)
        theta = torch.acos(cosine)
        m_arc = self.m * margin_scaler * -1  # negative for high quality = larger margin
        theta_m = (theta + one_hot * m_arc.unsqueeze(1)).clamp(self.eps, math.pi - self.eps)
        cosine_m = torch.cos(theta_m)
        
        # Additive margin (g_additive)
        m_cos = self.m + (self.m * margin_scaler)
        cosine_m = cosine_m - one_hot * m_cos.unsqueeze(1)
        
        logits = cosine_m * self.s
        return logits, embeddings_norm


class ConvNeXtAdaFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, dropout=0.1):
        super().__init__()
        self.backbone = ConvNeXtBackbone(
            embedding_dim=embedding_dim, 
            pretrained=True,
            dropout=dropout
        )
        self.head = AdaFace(
            embedding_size=embedding_dim, 
            num_classes=num_classes, 
            s=64.0, 
            m=0.4,
            h=0.333,
            t_alpha=0.01
        )

    def forward(self, x, labels=None):
        emb = self.backbone(x)                    # (B, D) - raw embeddings
        if labels is not None:
            logits, emb_norm = self.head(emb, labels)  # (B, C), (B, D)
            return logits, emb_norm
        else:
            logits, emb_norm = self.head(emb, None)
            return emb_norm  # Return normalized embeddings for inference

### VI. Metrics

In [7]:
@torch.no_grad()
def top1_accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    correct = (preds == labels).sum().item()
    return correct / labels.size(0)

@torch.no_grad()
def topk_accuracy(logits, labels, k=5):
    topk = logits.topk(k, dim=1).indices
    correct = (topk == labels.view(-1, 1)).any(dim=1).sum().item()
    return correct / labels.size(0)


### VII. Train and Evaluation Loop

In [8]:
from torch.amp import autocast

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0.0
    total_top1 = 0.0
    n = 0
    
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Mixed precision forward pass
        with autocast():
            logits, _ = model(images, labels=labels)
            loss = criterion(logits, labels)
        
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        
        # Gradient clipping for stability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_top1 += top1_accuracy(logits, labels) * bs
        n += bs
        
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / n, total_top1 / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    n = 0

    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits, _ = model(images, labels=labels)  # still ok to pass labels for consistent logits
        loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_top1 += top1_accuracy(logits, labels) * bs
        total_top5 += topk_accuracy(logits, labels, k=5) * bs
        n += bs

    return total_loss / n, total_top1 / n, total_top5 / n


In [ ]:
from torch.amp import GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

model = ConvNeXtAdaFaceModel(
    num_classes=num_classes, 
    embedding_dim=512,
    dropout=0.1  # Regularization
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# You want to train BOTH backbone + head
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=1e-4,  # Lower than 3e-4
    weight_decay=5e-4  # Lower than 1e-2
)

EPOCHS = 20
best_val_top1 = -1.0
best_path = "best_convnext_adaface.pth"

warmup = LinearLR(optimizer, start_factor=0.1, total_iters=5)
cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS-5, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

history = []

scaler = GradScaler()
best_val_top1 = -1.0
patience = 10
patience_counter = 0

import time
from tqdm.auto import tqdm

model.train()

t0 = time.time()
for step, (images, labels) in enumerate(train_loader):
    print("Entered loop. Step:", step, "batch:", images.shape, "labels:", labels.shape)
    images = images.to(device)
    labels = labels.to(device)

    t_fwd0 = time.time()
    logits, emb = model(images, labels=labels)
    print("Forward done in", time.time() - t_fwd0, "sec", "logits:", logits.shape, "emb:", emb.shape)

    loss = criterion(logits, labels)
    print("Loss:", float(loss))

    t_bwd0 = time.time()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print("Backward+step done in", time.time() - t_bwd0, "sec")

    break

print("Total time:", time.time() - t0, "sec")


for epoch in range(1, EPOCHS + 1):
    train_loss, train_top1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )
    val_loss, val_top1, val_top5 = evaluate(model, val_loader, criterion, device)
    
    # Step scheduler
    scheduler.step()
    
    print(f"Epoch {epoch:02d} | "
          f"LR={optimizer.param_groups[0]['lr']:.2e} | "
          f"train_loss={train_loss:.4f} train_top1={train_top1*100:.2f}% | "
          f"val_loss={val_loss:.4f} val_top1={val_top1*100:.2f}% val_top5={val_top5*100:.2f}%")
    
    # Save best
    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "num_classes": num_classes,
            "best_val_top1": best_val_top1,
            "label_to_index": label_to_index,
            "index_to_label": index_to_label,
        }, best_path)
        print(f"✅ Saved best model (val_top1={best_val_top1*100:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break


### VIII. Test Evaluation

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])

test_loss, test_top1, test_top5 = evaluate(model, test_loader, criterion, device)
print(f"TEST | loss={test_loss:.4f} top1={test_top1*100:.2f}% top5={test_top5*100:.2f}%")

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    """Extract normalized embeddings for all samples"""
    model.eval()
    all_embeddings = []
    all_labels = []
    
    for images, labels in tqdm(loader, desc="Extracting", leave=False):
        images = images.to(device, non_blocking=True)
        embeddings = model(images, labels=None)  # Get normalized embeddings
        all_embeddings.append(embeddings.cpu())
        all_labels.append(labels)
    
    return torch.cat(all_embeddings), torch.cat(all_labels)


@torch.no_grad()
def compute_reid_metrics(query_emb, query_labels, gallery_emb, gallery_labels):
    """
    Compute Re-ID metrics: mAP and CMC
    """
    # Cosine similarity
    similarity = torch.mm(query_emb, gallery_emb.t())  # (Q, G)
    
    # Sort by similarity (descending)
    indices = torch.argsort(similarity, dim=1, descending=True)
    
    # CMC (Cumulative Matching Characteristics)
    cmc = torch.zeros(len(gallery_labels))
    aps = []
    
    for i, query_label in enumerate(query_labels):
        # Get retrieval order
        retrieval_labels = gallery_labels[indices[i]]
        
        # Find matches
        matches = (retrieval_labels == query_label)
        
        # CMC: first match position
        if matches.any():
            first_match = matches.nonzero(as_tuple=True)[0][0]
            cmc[first_match:] += 1
        
        # AP (Average Precision)
        if matches.sum() > 0:
            cumsum = torch.cumsum(matches.float(), dim=0)
            precision = cumsum / torch.arange(1, len(matches) + 1, device=matches.device)
            ap = (precision * matches.float()).sum() / matches.sum()
            aps.append(ap.item())
    
    cmc = cmc / len(query_labels)
    mAP = np.mean(aps) if aps else 0.0
    
    return {
        'mAP': mAP * 100,
        'Rank-1': cmc[0].item() * 100,
        'Rank-5': cmc[4].item() * 100 if len(cmc) > 4 else 0.0,
        'Rank-10': cmc[9].item() * 100 if len(cmc) > 9 else 0.0,
    }


# After training, evaluate on test set
print("\n" + "="*60)
print("Final Evaluation on Test Set")
print("="*60)

test_emb, test_labels = extract_embeddings(model, test_loader, device)
metrics = compute_reid_metrics(test_emb, test_labels, test_emb, test_labels)

print(f"mAP:    {metrics['mAP']:.2f}%")
print(f"Rank-1: {metrics['Rank-1']:.2f}%")
print(f"Rank-5: {metrics['Rank-5']:.2f}%")
print(f"Rank-10: {metrics['Rank-10']:.2f}%")

### IX. t-SNE After Training

In [ ]:
from sklearn.manifold import TSNE

@torch.no_grad()
def extract_embeddings(model, loader, device, max_points=2000):
    model.eval()
    all_emb = []
    all_lab = []

    count = 0
    for images, labels in tqdm(loader, desc="Embed"):
        images = images.to(device, non_blocking=True)
        _, emb = model(images, labels=None)   # inference mode
        all_emb.append(emb.cpu().numpy())
        all_lab.append(labels.numpy())
        count += labels.size(0)
        if count >= max_points:
            break

    emb = np.concatenate(all_emb, axis=0)[:max_points]
    lab = np.concatenate(all_lab, axis=0)[:max_points]
    return emb, lab

emb, lab = extract_embeddings(model, test_loader, device, max_points=2000)
print("Embeddings:", emb.shape, "Labels:", lab.shape)


In [ ]:
tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42)
emb_2d = tsne.fit_transform(emb)

plt.figure(figsize=(10, 8))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], s=8, c=lab, alpha=0.7)
plt.title("t-SNE of ConvNeXt + AdaFace Embeddings (Test subset)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Class label (index)")
plt.show()